# train-eval-mode-branch — ex2: BatchNorm running stats freeze in eval — same input, two different outputs

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `train-eval-mode-branch`. Running the final beacon cell reports progress against the `PyTorch: train/eval mode` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: train/eval mode` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`train-eval-mode-branch`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "train-eval-mode-branch"
DD_SUBTOPIC = "PyTorch: train/eval mode"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `eval()` freezes BatchNorm running stats, not just Dropout

Ex1 toggled `.train()`/`.eval()` around Dropout. The deepening move is BatchNorm — a layer whose `train()` vs `eval()` behaviour is MORE consequential than Dropout's, because BN updates running stats in train mode and reads them in eval mode.

**Train mode (BN):**
1. Compute batch mean / variance over the current minibatch.
2. Normalize using those batch statistics.
3. Update `running_mean` / `running_var` via momentum.

**Eval mode (BN):**
1. Normalize using the saved `running_mean` / `running_var`.
2. Do NOT update them.

Consequence: feeding the SAME input through the SAME BN layer in the two modes gives DIFFERENT outputs (unless batch stats happen to equal running stats, which is the limit of training, not the general case).

```python
bn = nn.BatchNorm1d(4)
bn.train(); bn(x)               # uses batch stats, mutates running_*
bn.eval();  bn(x)               # uses running stats, no mutation
```

**Why eval mode also matters for Dropout — but differently.** Dropout in eval is identity (no-op); BN in eval is a fixed affine given by the running stats. Forgetting `.eval()` at inference time is the #1 silent bug in ARENA-scale codebases.

### Exercise 2 — BatchNorm running stats freeze in eval — same input, two different outputs

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the train→eval transition on a `BatchNorm1d` layer: do one train pass to populate `running_mean`/`running_var`, then run the SAME input in `eval()` and verify (a) the two outputs differ, (b) `running_mean` is unchanged across the eval pass.
> Keywords: batchnorm, train, eval, running_stats
> ```

**KCs targeted:** `batchnorm-mode-dependent-stats`, `running-mean-update-only-in-train`

Implement `ex2_bn_train_eval_divergence(x)`. Demonstrates how `.eval()` freezes BatchNorm's running statistics.

Input:
- `x`: `(B, C)` float tensor with `B >= 2` and `C` features.

Algorithm:
1. Construct `bn = nn.BatchNorm1d(C)` where `C = x.shape[1]`.
2. `bn.train()`; compute `out_train = bn(x)` (this updates `running_mean`/`running_var`).
3. Snapshot `rm_after_train = bn.running_mean.clone()`.
4. `bn.eval()`; compute `out_eval = bn(x)`.
5. Snapshot `rm_after_eval = bn.running_mean.clone()`.
6. Return a `dict` with:
   - `'out_train'`: the train-mode output (tensor)
   - `'out_eval'`: the eval-mode output (tensor)
   - `'diff_train_vs_eval'`: `(out_train - out_eval).abs().max().item()` (float)
   - `'rm_after_train'`: the running_mean snapshot AFTER train pass (tensor)
   - `'rm_after_eval'`: the running_mean snapshot AFTER eval pass (tensor)
   - `'rm_changed_in_eval'`: bool — True iff `rm_after_eval` differs from `rm_after_train` (it should be False).

Constraint: do NOT call `.no_grad()` or `.detach()`. Pure module behaviour.

The test verifies the divergence is real and that the running stats are frozen in eval.

In [ ]:
def ex2_bn_train_eval_divergence(x):
    C = x.shape[1]
    bn = nn.BatchNorm1d(C)

    bn.train()
    out_train = bn(x)
    rm_after_train = bn.running_mean.clone()

    bn.eval()
    out_eval = bn(x)
    rm_after_eval = bn.running_mean.clone()

    diff = (out_train - out_eval).abs().max().item()
    rm_changed = not t.equal(rm_after_train, rm_after_eval)

    return {
        'out_train': out_train,
        'out_eval': out_eval,
        'diff_train_vs_eval': float(diff),
        'rm_after_train': rm_after_train,
        'rm_after_eval': rm_after_eval,
        'rm_changed_in_eval': bool(rm_changed),
    }


<details><summary>Solution</summary>

```python
def ex2_bn_train_eval_divergence(x):
    C = x.shape[1]
    bn = nn.BatchNorm1d(C)

    bn.train()
    out_train = bn(x)
    rm_after_train = bn.running_mean.clone()

    bn.eval()
    out_eval = bn(x)
    rm_after_eval = bn.running_mean.clone()

    diff = (out_train - out_eval).abs().max().item()
    rm_changed = not t.equal(rm_after_train, rm_after_eval)

    return {
        'out_train': out_train,
        'out_eval': out_eval,
        'diff_train_vs_eval': float(diff),
        'rm_after_train': rm_after_train,
        'rm_after_eval': rm_after_eval,
        'rm_changed_in_eval': bool(rm_changed),
    }
```

**Train uses batch stats; eval uses running stats.** That's the whole story. After the FIRST train pass on data with mean ≠ 0, the running_mean has been updated toward that batch mean (by `momentum * batch_mean`, default `momentum=0.1`). The eval pass then normalizes using that partially-updated running stat — which is far from the actual batch stat — so the eval output diverges from the train output.

**`running_mean.clone()` before swapping modes.** `running_mean` is a registered buffer that BN may mutate again on subsequent train passes. Cloning takes a snapshot that's safe to compare after eval.

**Why this is the silent inference bug.** Forget to call `.eval()` at inference, and BN keeps updating running stats from your evaluation data — corrupting them. The corruption compounds across eval batches. The fix is one line; the training-loop discipline is the actual lesson.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()